# 🚀 Notebook 6: Full Pipeline — ORNJ PDFs to OWL Ontology

End-to-end: 16 ORNJ classification PDFs → Text → NLP/LLM Extraction → ORN-O Ontology → Validation

---

In [ ]:
!pip install -q PyMuPDF pdfplumber pytesseract Pillow spacy scikit-learn owlready2 rdflib networkx matplotlib anthropic tqdm
!apt-get install -q tesseract-ocr > /dev/null 2>&1
!python -m spacy download en_core_web_sm -q
!git clone https://github.com/YOUR_USERNAME/orn-ontology-builder.git 2>/dev/null || true
import sys; sys.path.insert(0, 'orn-ontology-builder')
import os, json, logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
os.makedirs('output', exist_ok=True)
print('Setup complete!')

## 1. Upload ORNJ PDFs

In [ ]:
from google.colab import files
os.makedirs('pdfs', exist_ok=True)
print('Upload your 16 ORNJ classification PDFs:')
uploaded = files.upload()
for fn, content in uploaded.items():
    with open(f'pdfs/{fn}', 'wb') as f:
        f.write(content)

## 2. Configuration

In [ ]:
CONFIG = {
    'pdf_directory': 'pdfs/',
    'method': 'hybrid',  # 'nlp', 'llm', or 'hybrid'
    'spacy_model': 'en_core_web_sm',
    'min_concept_frequency': 2,
    'max_concepts': 300,
    'min_relation_confidence': 0.3,
    'anthropic_api_key': '',  # Set your key or use env var
    'llm_model': 'claude-sonnet-4-20250514',
    'domain_context': 'osteoradionecrosis of the jaw classification and staging systems',
    'ontology_iri': 'http://example.org/orn-ontology',
    'output_path': 'output/orn_ontology.owl',
    'run_reasoner': True,
}

## 3. Extract Text

In [ ]:
from src.pdf_extractor import PDFExtractor
extractor = PDFExtractor(ocr_enabled=True)
documents = extractor.extract_from_directory(CONFIG['pdf_directory'])
print(f'Extracted {len(documents)} documents')

## 4. Preprocess

In [ ]:
from src.preprocessor import TextPreprocessor
preprocessor = TextPreprocessor()
segments = preprocessor.preprocess_documents(documents)
print(f'Total segments: {len(segments)}')

## 5. Extract Knowledge

In [ ]:
from src.concept_extractor import ConceptExtractor
from src.relation_extractor import RelationExtractor

concepts, relations, llm_knowledge = [], [], None

if CONFIG['method'] in ('nlp', 'hybrid'):
    print('NLP extraction...')
    concepts = ConceptExtractor(min_frequency=CONFIG['min_concept_frequency']).extract(segments)
    relations = RelationExtractor(min_confidence=CONFIG['min_relation_confidence']).extract(segments, concepts)
    print(f'  NLP: {len(concepts)} concepts, {len(relations)} relations')

if CONFIG['method'] in ('llm', 'hybrid'):
    api_key = CONFIG['anthropic_api_key'] or os.environ.get('ANTHROPIC_API_KEY')
    if api_key:
        print('LLM extraction...')
        from src.llm_extractor import LLMExtractor
        llm_ext = LLMExtractor(api_key=api_key, domain_context=CONFIG['domain_context'])
        llm_knowledge = llm_ext.extract_from_documents(documents)
        print(f'  LLM: {len(llm_knowledge.concepts)} concepts, {len(llm_knowledge.relations)} relations')
    else:
        print('  Skipping LLM (no API key)')

## 6. Build Ontology

In [ ]:
from src.ontology_builder import OntologyBuilder
builder = OntologyBuilder(iri=CONFIG['ontology_iri'])
if concepts: builder.add_concepts(concepts)
if relations: builder.add_relations(relations)
if llm_knowledge: builder.from_llm_output(llm_knowledge)
builder.save(CONFIG['output_path'])
stats = builder.get_stats()
print('\nORN-O Built!')
for k, v in stats.items(): print(f'  {k}: {v}')

## 7. Validate

In [ ]:
if CONFIG['run_reasoner']:
    from src.validator import OntologyValidator
    report = OntologyValidator(reasoner='hermit').validate(builder.onto)
    print(report.summary())

## 8. Query

In [ ]:
from src.query_engine import QueryEngine
engine = QueryEngine(CONFIG['output_path'])
print(f'Loaded {engine.count_triples()} triples')
for c in engine.get_all_classes()[:20]:
    print(f'  {c["class"].split("/")[-1].split("#")[-1]}')

## 9. Visualize

In [ ]:
from src.visualizer import OntologyVisualizer
viz = OntologyVisualizer(CONFIG['output_path'])
viz.plot_hierarchy(figsize=(18,14), title='ORN-O Ontology', save_path='output/orn_graph.png')

## 10. Download

In [ ]:
from google.colab import files
files.download(CONFIG['output_path'])
print('Done!')

---
## 🎉 Done!

You now have the ORN-O ontology. Open it in [Protégé](https://protege.stanford.edu/) or upload the .ttl to [WebVOWL](http://www.visualdataweb.de/webvowl/).